# 2.2. Training: Pathway signatures
# 2.2. 模型训练：通路特征

### 📘 Overview
### 📘 概述

This notebook uses DESeq2 differential-expression results to perform pathway enrichment analysis with WikiPathways gene sets, producing –log₁₀ *p*-values that quantify pathway activation for each compound.
这个notebook使用DESeq2差异表达结果，通过WikiPathways基因集进行通路富集分析，生成–log₁₀ p值来量化每个化合物的通路激活程度。

**Inputs**  
**输入数据**  
DESeq2 differential expression data
DESeq2差异表达数据

**Output**  
**输出**  
An AnnData file with –log₁₀ *p*-values for each pathway–compound pair
包含每个通路-化合物对的–log₁₀ p值的AnnData文件

### ⚠️ 重要提示：专有数据访问

**请注意**：此 notebook 需要 `training_data_deseq2.h5ad` 文件，该文件通常是专有的，受知识产权限制保护。

**依赖关系**：
- 此 notebook 依赖于 `2.1_Training_Gene_Signatures.ipynb` 的输出
- 如果 `2.1` 已运行并生成本地文件，代码会自动尝试加载

**如果您遇到 S3 访问错误**：
1. **申请数据访问权限**：请联系 DILImap@cellarity.com 或访问项目 GitHub Issues 页面
2. **使用本地数据**：如果您已有本地数据文件，代码会自动尝试从以下路径加载：
   - `training_data_deseq2.h5ad`
   - `../data/training_data_deseq2.h5ad`
   - `./data/training_data_deseq2.h5ad`
   - `../2.1_Training_Gene_Signatures/training_data_deseq2.h5ad`
3. **检查访问权限**：如果您已有访问权限，请检查 S3 凭证配置和网络连接

**Note**  
Please note that raw training data files are proprietary due to IP restrictions and will only be shared upon request, subject to data-sharing agreements and institutional policies.

In [1]:
%%capture

!conda install -c bioconda gseapy

In [2]:
import numpy as np


import dilimap as dmap

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
dmap.logging.print_version()

Running dilimap 0.2.dev20+gc285bd842 (python 3.10.19) on 2025-11-20 01:28.


## Run pathway enrichment analysis
## 运行通路富集分析

使用WikiPathways基因集对DESeq2结果进行通路富集分析，计算每个通路的激活分数。

In [ ]:
# 尝试从 S3 读取专有训练数据
# 注意：这些数据是专有的，需要数据共享协议才能访问
# 如果遇到访问错误，请参考下面的错误处理说明

try:
    adata_deseq = dmap.s3.read('training_data_deseq2.h5ad', package_name='proprietary/data')
    print("✓ 成功从 S3 加载数据")
except Exception as e:
    error_msg = str(e)
    print("=" * 80)
    print("❌ 无法从 S3 访问专有训练数据")
    print("=" * 80)
    print("\n错误信息：")
    print(f"  {error_msg}\n")
    print("原因：")
    print("  这些训练数据文件是专有的，受知识产权限制保护。")
    print("  只有在签署数据共享协议并符合机构政策后才会共享。\n")
    print("解决方案：")
    print("  1. 如果您已有数据访问权限，请检查：")
    print("     - S3 凭证是否正确配置")
    print("     - 网络连接是否正常")
    print("     - 是否已签署数据共享协议\n")
    print("  2. 如果您需要申请数据访问权限，请联系：")
    print("     - DILImap@cellarity.com")
    print("     - 或访问项目 GitHub Issues 页面\n")
    print("  3. 如果您已有本地数据文件，可以使用以下代码加载：")
    print("     import anndata as ad")
    print("     adata_deseq = ad.read_h5ad('path/to/your/local/training_data_deseq2.h5ad')\n")
    print("  4. 注意：此 notebook 依赖于 2.1_Training_Gene_Signatures.ipynb 的输出")
    print("     如果 2.1 已运行并生成本地文件，请使用该文件\n")
    print("=" * 80)
    
    # 尝试从本地文件加载（如果存在）
    import os
    import anndata as ad
    local_paths = [
        'training_data_deseq2.h5ad',
        '../data/training_data_deseq2.h5ad',
        './data/training_data_deseq2.h5ad',
        '../2.1_Training_Gene_Signatures/training_data_deseq2.h5ad',
    ]
    
    loaded = False
    for path in local_paths:
        if os.path.exists(path):
            try:
                print(f"\n尝试从本地路径加载: {path}")
                adata_deseq = ad.read_h5ad(path)
                print(f"✓ 成功从本地文件加载数据: {path}")
                loaded = True
                break
            except Exception as local_e:
                print(f"  ✗ 无法从 {path} 加载: {local_e}")
    
    if not loaded:
        raise RuntimeError(
            "无法加载训练数据。请申请数据访问权限或提供本地数据文件。\n"
            "联系邮箱: DILImap@cellarity.com\n"
            "注意: 此文件通常由 2.1_Training_Gene_Signatures.ipynb 生成"
        )

Package: s3://dilimap/proprietary/data. Top hash: ede9d7c164


ClientError: An error occurred (NoSuchVersion) when calling the GetObject operation: The specified version does not exist.

In [ ]:
FDR = adata_deseq.to_df('FDR')
# 从DESeq2结果中提取调整后的p值（FDR = False Discovery Rate，错误发现率）

NameError: name 'adata_deseq' is not defined

In [ ]:
adata_wiki = dmap.pp.pathway_signatures(FDR)
# 计算通路特征：使用WikiPathways基因集进行富集分析
# 输入：调整后的p值矩阵（基因 × 化合物）
# 输出：通路激活分数矩阵（通路 × 化合物），值为–log₁₀ p值

In [ ]:
adata_wiki.obs = adata_deseq.obs.copy()

Package: s3://compbio-analysis-prod/models/DILImap. Top hash: fa83af3701


/srv/conda/envs/saturn/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Mapping metadata and clinical annotations

In [7]:
## Simplify keys

adata_wiki.obs.rename(
    columns={
        'CONCENTRATION_UM': 'dose_uM',
        'DOSE_LEVEL': 'dose_level',
        'COMPOUND': 'compound_name',
    },
    inplace=True,
)

adata_wiki.obs['compound_name'] = np.where(
    adata_wiki.obs['compound_name'].isna(),
    adata_wiki.obs_names,
    adata_wiki.obs['compound_name'],
)

adata_wiki.obs_names = adata_wiki.obs_names.str.replace('CPZ', 'Chlorpromazine')
adata_wiki.obs.loc[adata_wiki.obs['compound_name'] == 'Ibrutinib', 'SPLIT'] = 'training'

NameError: name 'adata_wiki' is not defined

In [8]:
## Cmax annotations
obs_names = adata_wiki.obs_names.str.lower()

df_CMAX = dmap.s3.read('compound_Cmax_values.csv')
df_CMAX.index = df_CMAX.index.str.lower()
adata_wiki.obs['Cmax_uM'] = obs_names.map(df_CMAX['Cmax_median'])

## DILI annotations
df_DILI = dmap.s3.read('compound_DILI_labels.csv')
df_DILI.index = df_DILI.index.str.lower()
for col in df_DILI.columns:
    adata_wiki.obs[col] = obs_names.map(df_DILI[col])

adata_wiki.obs['DILIrank'] = adata_wiki.obs['DILIrank'].replace(np.nan, '')
adata_wiki.obs['livertox_score'] = adata_wiki.obs['livertox_score'].replace(np.nan, '')

## Viability (LDH) IC10 annotations
df_LDH = dmap.s3.read('compound_cell_viability_IC10.csv')
df_LDH.index = df_LDH.index.str.lower()
for k in df_LDH.columns:
    adata_wiki.obs[f'LDH_{k}'] = obs_names.map(df_LDH[k])

## Number DEGs
adata_wiki.obs['n_DEG'] = adata_deseq.obs['n_DEG'] = (
    adata_deseq.layers['FDR'] < 0.05
).sum(1)

NameError: name 'adata_wiki' is not defined

In [9]:
set(
    adata_wiki.obs_names[
        adata_wiki.obs['Cmax_uM'].isna() | adata_wiki.obs['DILI_label'].isna()
    ]
)

NameError: name 'adata_wiki' is not defined

In [ ]:
adata_wiki = adata_wiki[
    ~(adata_wiki.obs['Cmax_uM'].isna() | adata_wiki.obs['DILI_label'].isna())
].copy()

/srv/conda/envs/saturn/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
adata_wiki.X = np.nan_to_num(adata_wiki.X)  # set nans to zero

## Push file to S3

In [ ]:
# dmap.s3.write(adata_wiki, 'training_data_pathways.h5ad', package_name='public/data')